## **Import Library**

In [1]:
import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from difflib import get_close_matches

## **Load Dataset**

In [2]:
dataset = pd.read_csv('../data/preprocessed/pre_movies.csv')

In [3]:
dataset.head(2)

,movie_id,title,genres,keywords,overview,cast,director,vote_count,vote_average,combined_features,score
0,19995,Avatar,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","['culture clash', 'future', 'space war', 'spac...","in the 22nd century, a parapleg marin is dispa...","['Sam Worthington', 'Zoe Saldana', 'Sigourney ...",['James Cameron'],11800,7.2,Action Adventure Fantasy Science Fiction cultu...,7.134875
1,285,Pirates of the Caribbean: At World's End,"['Adventure', 'Fantasy', 'Action']","['ocean', 'drug abuse', 'exotic island', 'east...","captain barbossa, long believ to be dead, ha c...","['Johnny Depp', 'Orlando Bloom', 'Keira Knight...",['Gore Verbinski'],4500,6.9,Adventure Fantasy Action ocean drug abuse exot...,6.786315


## **Feature Engginering**

### **Vectorization**

To convert str to numerical data
`stop_word='english'` to remove words that do not contribute to content similarity.

Output shape: (4803, 28391)
- 4803 = Number of movies
- 28391 = Number of unique words

In [4]:
vectorizer = TfidfVectorizer(stop_words='english')
x = vectorizer.fit_transform(dataset['combined_features'])
x.shape

(4803, 27216)

### **Cosine Similarity Matrix**
Calculates the similarity between all movies

In [5]:
cosine_sim = cosine_similarity(x, x)
cosine_sim.shape

(4803, 4803)

In [6]:
cosine_sim

array([[1.        , 0.01797277, 0.01882855, ..., 0.00690594, 0.02385733,
        0.        ],
       [0.01797277, 1.        , 0.00807044, ..., 0.01398067, 0.        ,
        0.00353715],
       [0.01882855, 0.00807044, 1.        , ..., 0.00916208, 0.05025797,
        0.        ],
       ...,
       [0.00690594, 0.01398067, 0.00916208, ..., 1.        , 0.0151971 ,
        0.01547518],
       [0.02385733, 0.        , 0.05025797, ..., 0.0151971 , 1.        ,
        0.01092618],
       [0.        , 0.00353715, 0.        , ..., 0.01547518, 0.01092618,
        1.        ]], shape=(4803, 4803))

### **get_recommendations**

To retrieve movie recommendations

use fuzzy matching with `get_close_matches` to tolerate typos.

In [7]:
indices = pd.Series(dataset.index, index=dataset['title'])
indices = indices[~indices.index.duplicated(keep='last')]

def get_recommendations(title, cosine_sim=cosine_sim):
    close_matches = get_close_matches(title, dataset['title'], n=1, cutoff=0.6)
    if not close_matches:
        return 'Movie not found in the dataset.'
    
    matched_title = close_matches[0]
    print(f'Show Recommendations for: {matched_title}')
    
    idx = indices[matched_title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x:x[1], reverse=True)
    sim_scores = sim_scores[1:21]
    movie_indices = [i[0] for i in sim_scores]
    
    candidates = dataset.iloc[movie_indices][['movie_id', 'title', 'vote_count', 'vote_average', 'score']].copy()
    candidates['similarity'] = [i[1] for i in sim_scores]
    candidates['final_score'] = (candidates['similarity'] * 0.7) + (candidates['score'] * 0.3)
    
    return candidates.sort_values('final_score', ascending=False).head(10)

In [8]:
get_recommendations('superman')

Show Recommendations for: Superman


,movie_id,title,vote_count,vote_average,score,similarity,final_score
65,155,The Dark Knight,12002,8.2,8.078054,0.132990,2.516509
119,272,Batman Begins,7359,7.5,7.371842,0.146857,2.314352
16,24428,The Avengers,11776,7.4,7.322971,0.117289,2.278994
101,49538,X-Men: First Class,5181,7.1,6.974490,0.157357,2.202497
1720,23483,Kick-Ass,4645,7.1,6.961990,0.145112,2.190175
511,36657,X-Men,4097,6.8,6.692083,0.171150,2.127430
870,8536,Superman II,629,6.5,6.279964,0.345587,2.125900
1359,268,Batman,2096,7.0,6.763830,0.129408,2.119735
14,49521,Man of Steel,6359,6.5,6.457642,0.234196,2.101230
30,558,Spider-Man 2,4321,6.7,6.611433,0.132414,2.076120


## **Save Metrix**

In [9]:
with open('../model/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)